In [1]:
import pandas as pd

In [2]:
# Load patient and sample data
# skip the first 4 rows because they are usually descriptions, not data

patient_data = pd.read_csv('data_clinical_patient.txt', sep='\t', skiprows=4)
sample_data = pd.read_csv('data_clinical_sample.txt', sep='\t', skiprows=4)

In [3]:
# Merge patient_data and sample_data on PATIENT_ID

clinical_full = pd.merge(patient_data, sample_data, on='PATIENT_ID')

In [4]:
# keep the relevant columns needed for causal inference

clinical_subset = clinical_full[['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', 'CHEMOTHERAPY', 'AGE_AT_DIAGNOSIS']]

In [5]:
# Load the gene expression data
genetic_data = pd.read_csv('data_mrna_illumina_microarray.txt', sep='\t')

# The file has genes as rows, but we need them as columns for ML
genetic_data = genetic_data.set_index('Hugo_Symbol').drop('Entrez_Gene_Id', axis=1).T
genetic_data.index.name = 'PATIENT_ID'
genetic_data = genetic_data.reset_index()

# Merge everything into one final dataframe
final_df = pd.merge(clinical_subset, genetic_data, on='PATIENT_ID')

In [6]:
# Drop missing values
final_df = final_df.dropna()

# Create a copy of the dataframe to break the link to old data
final_df = final_df.copy()

# Use .loc to modify columns safely
final_df.loc[:, 'OS_STATUS'] = final_df['OS_STATUS'].astype(str).str[0].astype(int)
final_df.loc[:, 'CHEMOTHERAPY'] = final_df['CHEMOTHERAPY'].map({'YES': 1, 'NO': 0})

In [7]:
final_df

,PATIENT_ID,OS_STATUS,OS_MONTHS,CHEMOTHERAPY,AGE_AT_DIAGNOSIS,RERE,RNF165,PHF7,CIDEA,TENT2,...,SBF2-AS1,VN1R4,TRPV5,UGGT1,CR590356,VPS72,CSMD3,CC2D1A,IGSF9,FAM71A
0,MB-0000,0,140.500000,0,75.65,9.738092,6.469688,5.652674,11.558869,8.340484,...,5.295203,5.474224,5.415184,7.021679,5.947334,8.010657,5.299815,6.235804,5.947404,5.133576
1,MB-0002,0,84.633333,0,43.19,9.013876,5.748717,5.611212,6.199492,8.341091,...,5.442257,5.303871,5.507905,7.612797,5.519225,7.988643,5.194247,6.328059,6.938685,5.604560
2,MB-0005,1,163.700000,1,48.87,7.963493,5.553056,5.793398,6.489781,8.862815,...,5.368716,5.370394,5.314009,6.619220,5.496590,8.044471,5.306294,6.306927,7.397672,5.645597
3,MB-0006,0,164.933333,1,47.68,8.177157,5.391160,5.807604,5.319779,8.693784,...,5.295628,5.694697,5.410028,6.966482,5.600679,7.527300,5.197607,6.515638,6.175716,5.354582
4,MB-0008,1,41.366667,1,76.97,8.050127,5.530582,5.934570,8.787583,8.055626,...,5.192858,5.474929,5.389733,6.942461,6.707482,8.115359,5.254136,6.323751,6.272568,5.030636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,MB-7295,0,196.866667,0,43.10,8.589374,6.002483,5.802742,5.474550,8.251690,...,5.240904,5.521765,5.646490,7.536208,5.824289,8.314500,5.337300,6.278034,6.923887,5.168953
1976,MB-7296,1,44.733333,0,42.88,8.402660,6.104059,5.256086,6.659117,8.641838,...,5.171671,5.441341,5.612519,7.709596,5.899345,8.105717,5.397931,6.325456,6.524268,5.252479
1977,MB-7297,1,175.966667,0,62.90,8.236918,5.402870,5.571897,5.439574,8.979375,...,5.182920,5.339665,5.573512,7.729912,5.588047,7.944622,5.412713,6.254337,6.121864,5.357823
1978,MB-7298,1,86.233333,0,61.16,8.376571,5.617954,5.631592,5.734358,8.628511,...,5.351516,5.361063,5.436273,7.329023,5.713436,8.348807,5.474224,6.415853,7.029076,5.512290


In [8]:
# Seperate clinical variables from gene expression data
clinical_cols = ['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', 'CHEMOTHERAPY', 'AGE_AT_DIAGNOSIS']
gene_cols = final_df.columns.drop(clinical_cols)

# Calculate the variance for each gene and pick the top 500
# High variance means the gene expression varies significantly across patients
top_500_genes = final_df[gene_cols].var().sort_values(ascending=False).head(500).index.tolist()

# Create the dataframe
df = final_df[clinical_cols + top_500_genes]

In [9]:
df

,PATIENT_ID,OS_STATUS,OS_MONTHS,CHEMOTHERAPY,AGE_AT_DIAGNOSIS,SCGB2A2,MUCL1,SCGB1D2,PIP,TFF1,...,NKX3-1,LINC01116,H2BC21,KRT18,GASK1B,GAS1,QDPR,CNKSR3,RNASE4,MFAP5
0,MB-0000,0,140.500000,0,75.65,12.019811,12.031687,10.439118,12.607582,8.039461,...,6.596974,7.331212,7.342041,10.656037,8.554209,10.092698,9.561828,7.307892,10.231346,10.712850
1,MB-0002,0,84.633333,0,43.19,5.751104,8.217465,5.481802,10.629072,7.486494,...,6.391452,8.743604,10.052989,12.389630,10.057009,9.051855,9.864041,6.439827,9.614031,9.704714
2,MB-0005,1,163.700000,1,48.87,13.492220,10.946228,9.987552,11.602012,9.355991,...,9.135396,8.010657,9.583128,11.338593,11.093532,9.859712,10.259575,6.059425,9.489386,10.489856
3,MB-0006,0,164.933333,1,47.68,13.327120,8.965470,9.665018,8.338585,13.731721,...,6.557337,7.758969,7.688023,13.349114,10.237150,9.919905,10.263959,6.881495,7.279506,10.720184
4,MB-0008,1,41.366667,1,76.97,8.876743,6.476973,7.038982,8.384237,8.683432,...,6.736151,9.104646,10.288807,13.225869,8.696859,8.544386,9.577056,6.026244,8.616926,7.930213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,MB-7295,0,196.866667,0,43.10,11.931463,6.042132,7.766011,12.853219,9.150302,...,7.650082,8.811466,7.072031,12.162168,9.672214,10.005950,9.432089,6.800736,9.217418,10.365286
1976,MB-7296,1,44.733333,0,42.88,13.973004,7.215314,13.230864,7.701697,12.752417,...,5.936503,8.399984,8.197288,12.189163,9.221997,8.043926,10.948664,7.135552,7.550066,8.607071
1977,MB-7297,1,175.966667,0,62.90,8.006986,6.412046,6.110946,8.919217,7.237071,...,8.024375,8.928698,8.996347,13.059220,8.665807,8.147846,9.990066,6.168361,7.466155,8.187168
1978,MB-7298,1,86.233333,0,61.16,6.402666,8.794891,5.770089,10.137608,11.311807,...,7.537188,8.781717,9.787720,13.143164,8.282374,7.804472,8.620132,5.918426,7.792612,8.986533


In [10]:
# Find duplicate rows

df.duplicated().any()

np.False_

In [11]:
# Find duplicate columns
df.T.duplicated().any()

np.True_

In [12]:
# Remove columns that are exact duplicates of each other
df = df.T.drop_duplicates().T

In [13]:
df

,PATIENT_ID,OS_STATUS,OS_MONTHS,CHEMOTHERAPY,AGE_AT_DIAGNOSIS,SCGB2A2,MUCL1,SCGB1D2,PIP,TFF1,...,NKX3-1,LINC01116,H2BC21,KRT18,GASK1B,GAS1,QDPR,CNKSR3,RNASE4,MFAP5
0,MB-0000,0,140.5,0,75.65,12.019811,12.031687,10.439118,12.607582,8.039461,...,6.596974,7.331212,7.342041,10.656037,8.554209,10.092698,9.561828,7.307892,10.231346,10.71285
1,MB-0002,0,84.633333,0,43.19,5.751104,8.217465,5.481802,10.629072,7.486494,...,6.391452,8.743604,10.052989,12.38963,10.057009,9.051855,9.864041,6.439827,9.614031,9.704714
2,MB-0005,1,163.7,1,48.87,13.49222,10.946228,9.987552,11.602012,9.355991,...,9.135396,8.010657,9.583128,11.338593,11.093532,9.859712,10.259575,6.059425,9.489386,10.489856
3,MB-0006,0,164.933333,1,47.68,13.32712,8.96547,9.665018,8.338585,13.731721,...,6.557337,7.758969,7.688023,13.349114,10.23715,9.919905,10.263959,6.881495,7.279506,10.720184
4,MB-0008,1,41.366667,1,76.97,8.876743,6.476973,7.038982,8.384237,8.683432,...,6.736151,9.104646,10.288807,13.225869,8.696859,8.544386,9.577056,6.026244,8.616926,7.930213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,MB-7295,0,196.866667,0,43.1,11.931463,6.042132,7.766011,12.853219,9.150302,...,7.650082,8.811466,7.072031,12.162168,9.672214,10.00595,9.432089,6.800736,9.217418,10.365286
1976,MB-7296,1,44.733333,0,42.88,13.973004,7.215314,13.230864,7.701697,12.752417,...,5.936503,8.399984,8.197288,12.189163,9.221997,8.043926,10.948664,7.135552,7.550066,8.607071
1977,MB-7297,1,175.966667,0,62.9,8.006986,6.412046,6.110946,8.919217,7.237071,...,8.024375,8.928698,8.996347,13.05922,8.665807,8.147846,9.990066,6.168361,7.466155,8.187168
1978,MB-7298,1,86.233333,0,61.16,6.402666,8.794891,5.770089,10.137608,11.311807,...,7.537188,8.781717,9.78772,13.143164,8.282374,7.804472,8.620132,5.918426,7.792612,8.986533


In [14]:
# This will count how many unique columns the dataset has
print(f"Unique Column Names: {df.columns.nunique()}")

# This will show you exactly which names are repeated
duplicate_cols = df.columns[df.columns.duplicated()].tolist()
print(f"Repeated Column Names: {duplicate_cols}")

Unique Column Names: 496
Repeated Column Names: ['IGKC', 'IGHG1', 'MZB1', 'IL23A', 'IL23A', 'IL23A', 'IL23A', 'MT1E', 'MT1X']


In [15]:
# Create unique names for the duplicates

df.columns = [f'{col}_{i}' if i>0 else col for col, i in zip(df.columns, df.columns.to_series().groupby(df.columns).cumcount())]

In [16]:
# Verify the change
# This will count how many unique columns the dataset has
print(f"Unique Column Names: {df.columns.nunique()}")

# This will show you exactly which names are repeated
duplicate_cols = df.columns[df.columns.duplicated()].tolist()
print(f"Repeated Column Names: {duplicate_cols}")

Unique Column Names: 505
Repeated Column Names: []


In [17]:
#Verify the change

df.filter(regex='^IL23A')

,IL23A,IL23A_1,IL23A_2,IL23A_3,IL23A_4
0,7.395919,5.514902,5.463349,5.663504,5.457527
1,6.436743,5.455769,5.523949,5.546644,5.339594
2,6.33819,5.493733,5.228164,5.327659,5.372255
3,7.14332,5.563178,5.46984,5.800214,5.418735
4,7.594873,5.554849,5.483492,5.68751,5.492565
...,...,...,...,...,...
1975,7.148016,5.581225,5.43503,5.700858,5.701602
1976,7.927972,5.856614,5.562475,5.746283,5.628919
1977,8.040314,5.613772,5.984343,5.761156,5.853656
1978,9.107989,5.894399,6.122285,6.194237,5.685358


In [18]:
# Scaling

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols_to_scale = [col for col in df.columns if col not in ['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', 'CHEMOTHERAPY']]
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print("Successfully scaled Age and 500 gene features.")

Successfully scaled Age and 500 gene features.


In [19]:
df

,PATIENT_ID,OS_STATUS,OS_MONTHS,CHEMOTHERAPY,AGE_AT_DIAGNOSIS,SCGB2A2,MUCL1,SCGB1D2,PIP,TFF1,...,NKX3-1,LINC01116,H2BC21,KRT18,GASK1B,GAS1,QDPR,CNKSR3,RNASE4,MFAP5
0,MB-0000,0,140.5,0,1.121356,0.452604,0.962669,0.561161,0.766089,-0.363180,...,-0.066304,-0.970161,-0.919503,-0.990610,-0.388476,1.351291,0.052323,0.066663,1.838524,1.921041
1,MB-0002,0,84.633333,0,-1.383387,-1.588556,-0.319224,-1.130742,0.085141,-0.566000,...,-0.254601,0.324046,1.565828,0.599213,0.990553,0.395967,0.329802,-0.730452,1.271114,0.994321
2,MB-0005,1,163.7,1,-0.945096,0.932037,0.597865,0.407044,0.420000,0.119703,...,2.259369,-0.347570,1.135071,-0.364659,1.941709,1.137448,0.692964,-1.079763,1.156545,1.716055
3,MB-0006,0,164.933333,1,-1.036921,0.878278,-0.067833,0.296965,-0.703182,1.724654,...,-0.102619,-0.578197,-0.602315,1.479125,1.155859,1.192695,0.696989,-0.324883,-0.874686,1.927782
4,MB-0008,1,41.366667,1,1.223212,-0.570814,-0.904173,-0.599286,-0.687470,-0.126981,...,0.061208,0.654877,1.782021,1.366101,-0.257575,-0.069806,0.066305,-1.110232,0.354616,-0.636872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,MB-7295,0,196.866667,0,-1.390332,0.423837,-1.050315,-0.351155,0.850630,0.044259,...,0.898541,0.386230,-1.167041,0.390615,0.637450,1.271670,-0.066798,-0.399042,0.906563,1.601546
1976,MB-7296,1,44.733333,0,-1.407308,1.088585,-0.656029,1.513968,-0.922381,1.365460,...,-0.671420,0.009179,-0.135433,0.415371,0.224313,-0.529146,1.325656,-0.091591,-0.625998,-0.014677
1977,MB-7297,1,175.966667,0,0.137515,-0.854016,-0.925994,-0.916019,-0.503344,-0.657485,...,1.241465,0.493652,0.597125,1.213272,-0.286069,-0.433764,0.445512,-0.979730,-0.703126,-0.400669
1978,MB-7298,1,86.233333,0,0.003250,-1.376400,-0.125161,-1.032351,-0.084008,0.837066,...,0.795110,0.358970,1.322636,1.290255,-0.637923,-0.748926,-0.812302,-1.209237,-0.403061,0.334140


In [20]:
# Double machine learning

import doubleml as dml
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
df['CHEMOTHERAPY'] = df['CHEMOTHERAPY'].astype(int)
obj_dml_data = dml.DoubleMLData(df,
                            y_col = 'OS_MONTHS',
                            d_cols = 'CHEMOTHERAPY',
                            x_cols = [col for col in df.columns if col not in ['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', 'CHEMOTHERAPY']])
ml_l = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
ml_m = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
dml_plr_obj = dml.DoubleMLPLR(obj_dml_data, ml_l, ml_m)
dml_plr_obj.fit()
print(dml_plr_obj.summary)

                   coef   std err         t         P>|t|      2.5 %  \
CHEMOTHERAPY -26.947663  4.515478 -5.967842  2.404115e-09 -35.797838   

                 97.5 %  
CHEMOTHERAPY -18.097488  
